# S18 · Monte Carlo check: price the Reliance call by simulation

Notebook 2 priced our Reliance call at **₹40.81** with a formula we did not derive. Can we
trust it? Here we price the same call a second, completely different way, by doing
exactly what the key sentence says: **price = discounted average future payoff**.

We roll 100,000 possible prices for Reliance in a month, work out the call's payoff in
each, average them, and bring the average back to today. If the formula is right, the
two routes land on the same price. It is the same Monte Carlo engine as S17, on one
stock instead of five.


**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on each
  cell, top to bottom, and read the plain-English note above each one.
- New to the words (call, put, strike, premium)? Each one is explained the moment it
  appears, and all of them are in `primers/glossary.md`.
- New to the bell curve? `primers/distributions_and_the_bell_curve.md` is a ten-minute,
  picture-first read.
- Already confident with code or with finance? Look for the cells marked
  **Stretch (optional)** near the end.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy, pandas, matplotlib and scipy, which Colab already ships.
# So there is nothing to install here.
print("Setup complete - nothing to install.")

## Step 1 — get our tools ready

In [ ]:
from pathlib import Path
import numpy as np                 # fast maths, and random numbers
import pandas as pd                # tables of data
import matplotlib.pyplot as plt    # charts
from scipy.stats import norm       # the bell curve, for the formula we check against

print("Tools ready.")

## Step 2 — the same Reliance option as notebook 2

Load the saved prices, measure the volatility, and set the five inputs, exactly as before.

In [ ]:
# Where the saved prices live: next to this notebook, or on the course website.
local_copy = Path("../data/reliance_2023_2025.csv")
website_copy = ("https://girishmkulkarni.github.io/applied-maths-in-industry-site/"
                "sessions/S18/data/reliance_2023_2025.csv")

if local_copy.exists():
    prices = pd.read_csv(local_copy, index_col=0, parse_dates=True)
else:
    prices = pd.read_csv(website_copy, index_col=0, parse_dates=True)

print("days of prices:", len(prices))
print("from", prices.index[0].date(), "to", prices.index[-1].date())
prices.tail()

In [ ]:
daily = prices["Close"].loc["2025"].pct_change().dropna()

S = prices["Close"].iloc[-1]          # price on 31 Dec 2025
K = 1560                              # strike
T = 1 / 12                            # one month, in years
r = 0.06                              # 6% a year
sigma = daily.std() * np.sqrt(252)    # yearly volatility from 2025

print("S =", round(S, 2), "  K =", K, "  T =", round(T, 4), "  r =", r, "  sigma =", round(sigma, 3))

## Step 3 — the formula route, for comparison

The calculator from notebook 2, and the price we are aiming for.

In [ ]:
def black_scholes_price(S, K, T, r, sigma, kind="call"):
    """Fair price today of a European call or put (the Black-Scholes formula).

    S     : stock price today
    K     : strike price
    T     : time to expiry, in years (one month = 1/12)
    r     : interest rate, per year (6% = 0.06)
    sigma : volatility, per year (19.5% = 0.195)
    kind  : "call" or "put"
    """
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if kind == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


call = black_scholes_price(S, K, T, r, sigma, kind="call")
print("formula route: call price", round(call, 2))

## Step 4 — one possible future, by hand

How do we make up a realistic price for Reliance in a month? Draw one number `Z` from a
bell curve (average 0, spread 1) and plug it into this recipe:

$$ S_T = S \times e^{(r - \sigma^2/2)\,T \;+\; \sigma \sqrt{T}\, Z} $$

In words: returns follow a bell curve, and they compound. `Z = 0` is an ordinary month,
`Z = 1` a fairly good month, `Z = -1` a fairly bad one. The recipe grows at the interest rate
`r` on average; that choice is the "risk-neutral" idea explained in the session README.

In [ ]:
for Z in [-1.0, 0.0, 1.0]:
    one_future = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoff = max(one_future - K, 0)
    print("Z =", Z, " ->  Reliance in a month:", round(one_future, 2), "  call payoff:", round(payoff, 2))

## Step 5 — 100,000 possible futures

Now the same recipe with 100,000 random draws at once. The seed makes everyone's random
numbers identical, so your results match your neighbour's.

In [ ]:
np.random.seed(0)                     # same random futures for everyone
Z = np.random.standard_normal(100_000)
S_T = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

print("possible futures:", len(S_T))
print("lowest :", round(S_T.min(), 2))
print("highest:", round(S_T.max(), 2))
print("average:", round(S_T.mean(), 2))

plt.figure(figsize=(8, 5))
plt.hist(S_T, bins=80, color="#2E75B6")
plt.axvline(K, color="#C0392B", linestyle="--", label="strike 1,560")
plt.xlabel("Reliance in a month (Rs)")
plt.ylabel("how many futures")
plt.title("100,000 possible prices for Reliance in a month")
plt.legend()
plt.show()

## Step 6 — payoff, average, discount

The fair-price sentence, one line each: the call's payoff in every future, their average,
and the average brought back to today with `e^(-rT)`.

In [ ]:
payoffs = np.maximum(S_T - K, 0)            # call payoff in each future
average_payoff = payoffs.mean()
mc_price = np.exp(-r * T) * average_payoff   # average, then discount

print("futures where the call pays something:", round(100 * (payoffs > 0).mean(), 1), "%")
print("average payoff        :", round(average_payoff, 2))
print("simulation route price:", round(mc_price, 2))
print("formula route price   :", round(call, 2))

## Step 7 — more futures, closer answer

With few futures the answer wobbles; with more it settles onto the formula. This is the same
code as on the slide.

In [ ]:
path_counts = [1_000, 10_000, 100_000, 1_000_000]
estimates = []

np.random.seed(0)                      # same random futures for everyone
for n in path_counts:
    Z = np.random.standard_normal(n)   # n bell-curve draws
    S_T = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoffs = np.maximum(S_T - K, 0)   # call payoff in each future
    mc_price = np.exp(-r * T) * payoffs.mean()   # average, then discount
    estimates.append(mc_price)
    print(n, "futures  ->  call price", round(mc_price, 2))

print("formula route      ->  call price", round(call, 2))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(path_counts, estimates, marker="o", color="#2E75B6", linewidth=2, label="simulation route")
plt.axhline(call, color="#C0392B", linestyle="--", label="formula route = " + str(round(call, 2)))
plt.xscale("log")
plt.xlabel("number of futures (log scale)")
plt.ylabel("call price (Rs)")
plt.title("The simulation settles onto the formula price")
plt.legend()
plt.show()

The error shrinks like 1/√n: a hundred times more futures makes the answer about ten times
closer. When two independent methods agree, you can trust both pieces of code. That is how
professionals test pricing code.

### Stretch (optional) — an option with no neat formula

Skip this if the check above was enough. Simulation's real power is that you only change the
payoff line. Take a **cash-or-nothing** option: it pays a flat ₹100 if Reliance ends above
₹1,560, and nothing otherwise. Same futures, same average-then-discount.

In [ ]:
np.random.seed(0)
Z = np.random.standard_normal(1_000_000)
S_T = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

digital_payoffs = np.where(S_T > K, 100, 0)          # the only line that changed
digital_price = np.exp(-r * T) * digital_payoffs.mean()

print("cash-or-nothing option price:", round(digital_price, 2), "rupees")

### Stretch (optional) — are Reliance's real moves a bell curve?

Our simulation, and Black-Scholes itself, assume bell-curve returns. Check that against
Reliance's real daily moves over three years: count the days that moved more than three
standard deviations, and compare with what a bell curve expects.

In [ ]:
all_moves = prices["Close"].pct_change().dropna()
centre, spread = all_moves.mean(), all_moves.std()

big_days = (np.abs(all_moves - centre) > 3 * spread).sum()
expected = len(all_moves) * 2 * norm.sf(3)     # a bell curve's share beyond 3 sigma, both sides

print("days in 2023-2025          :", len(all_moves))
print("days beyond 3 sigma, real  :", big_days)
print("a bell curve expects about :", round(expected, 1))
print()
worst = all_moves.idxmin()
print("worst day:", worst.date(), " move", round(100 * all_moves.min(), 1), "%  =",
      round((all_moves.min() - centre) / spread, 1), "sigma")

More than four times as many big days as a bell curve allows, and one 6-sigma day: 4 June 2024,
the Lok Sabha results. Real markets have **fat tails** and **jumps**. That is why traders do
not use a single volatility for every strike, and why the slides call Black-Scholes a first
answer rather than the last word.

## Your turn

Price the **put** by simulation. Change the payoff line to the put's payoff,
`max(K - S_T, 0)`, use 1,000,000 futures, and compare your answer with the formula price from
`black_scholes_price(S, K, T, r, sigma, kind="put")`, about ₹29.86.

In [ ]:
# Your code here: simulate, put payoffs, average, discount, compare.

## What you just did

You priced the one-month Reliance call without the Black-Scholes formula: 100,000 made-up
months, a payoff in each, an average, one discount. The simulation route landed on the
formula route's ₹40.81, and got closer as you added futures. You also priced a contract that
has no neat formula, and saw that Reliance's real moves have fatter tails than the bell curve
both routes assume.

That is the whole session in three notebooks: what an option pays, what it is worth, and how
to check the answer.